In [5]:
import pandas as pd
import re

In [2]:
df = pd.read_csv("/Users/jamesantoarnoldj/Desktop/Projects/EE App/data/01-raw/ee_data.csv")
df.shape

(26993, 14)

In [3]:
df_clean = df.copy()

# 1. Remove exact duplicate rows:
before = len(df_clean)
df_clean.drop_duplicates(inplace=True)
print(f"Duplicates removed : {before - len(df_clean)}")

Duplicates removed : 546


In [6]:
# 2. Parse Date column:
# Some entries have a day of "0" (e.g. "0 May 2023") which are invalid.
# Replace day=0 with day=1 so pandas can parse them, then flag them.
def fix_date(val):
    if pd.isna(val):
        return pd.NaT
    val = str(val).strip()
    # Replace a leading "0 " day with "1 " so strptime can handle it
    val = re.sub(r'^0 ', '1 ', val)
    try:
        return pd.to_datetime(val, format="%d %b %Y")
    except Exception:
        return pd.NaT

df_clean["Date"] = df_clean["Date"].apply(fix_date)

unparseable = df_clean["Date"].isna().sum()
print(f"Unparseable dates (set to NaT) : {unparseable}")
print(f"Date range : {df_clean['Date'].min().date()} → {df_clean['Date'].max().date()}")

Unparseable dates (set to NaT) : 532
Date range : 2017-08-01 → 2023-09-09


In [7]:
# 3. Standardise Department:
# Strip trailing " Department" (case-insensitive) and whitespace
df_clean["Department"] = (
    df_clean["Department"]
    .str.strip()
    .str.replace(r"\s*department\s*$", "", case=False, regex=True)
    .str.strip()
    .fillna("Unknown")
)

print("Sample departments after cleaning:")
print(df_clean["Department"].value_counts().head(10))

Sample departments after cleaning:
Department
Software Development                 8936
Unknown                              4364
Quality Assurance and Testing        1942
Accounting & Taxation                1287
IT Consulting                        1017
Finance                               762
IT Network                            713
Business Intelligence & Analytics     661
IT Infrastructure Services            625
Technology / IT                       618
Name: count, dtype: int64


In [8]:
# 4. Clean Place:
# Many entries are "City, State" — extract just the city part.
# Also normalise common dual-name cities (e.g. "Bengaluru/Bangalore").
def extract_city(val):
    if pd.isna(val):
        return "Unknown"
    city = str(val).split(",")[0].strip()      # take part before first comma
    city = city.split("/")[0].strip()          # take part before first slash
    return city if city else "Unknown"

df_clean["Place"] = df_clean["Place"].apply(extract_city)

print("Top 10 cities after cleaning:")
print(df_clean["Place"].value_counts().head(10))

Top 10 cities after cleaning:
Place
Pune           3473
Bengaluru      3422
Bangalore      3184
Hyderabad      2731
Mumbai         2372
Unknown        1850
Chennai        1633
Navi Mumbai    1480
Kolkata        1201
Noida          1198
Name: count, dtype: int64


In [9]:
# 5. Fill categorical nulls:
df_clean["Job_type"] = df_clean["Job_type"].fillna("Unknown")
df_clean["Title"]    = df_clean["Title"].str.strip().fillna("Unknown")

# Text feedback columns → empty string (preserves rows for the chatbot context)
df_clean["Likes"]    = df_clean["Likes"].fillna("")
df_clean["Dislikes"] = df_clean["Dislikes"].fillna("")

print("Job_type distribution:")
print(df_clean["Job_type"].value_counts())

Job_type distribution:
Job_type
Unknown        14891
Full Time      11340
Contractual      131
Intern            51
Part Time         33
Freelancer         1
Name: count, dtype: int64


In [10]:
# 6. Fill numeric rating nulls with column median:
rating_cols = [
    "Overall_rating", "work_life_balance", "skill_development",
    "salary_and_benefits", "job_security", "career_growth", "work_satisfaction"
]

for col in rating_cols:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)

print("Missing values after fill:")
print(df_clean[rating_cols].isnull().sum())

Missing values after fill:
Overall_rating         0
work_life_balance      0
skill_development      0
salary_and_benefits    0
job_security           0
career_growth          0
work_satisfaction      0
dtype: int64


In [11]:
# 7. Final summary:
print(f"Shape  : {df_clean.shape}  (was {df.shape})")
print(f"\nRemaining nulls:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])
print("\nDtypes:")
print(df_clean.dtypes)


Shape  : (26447, 14)  (was (26993, 14))

Remaining nulls:
Date    532
dtype: int64

Dtypes:
Title                          object
Place                          object
Job_type                       object
Department                     object
Date                   datetime64[ns]
Overall_rating                float64
work_life_balance             float64
skill_development             float64
salary_and_benefits           float64
job_security                  float64
career_growth                 float64
work_satisfaction             float64
Likes                          object
Dislikes                       object
dtype: object


In [12]:
import os

out_dir = "/Users/jamesantoarnoldj/Desktop/Projects/EE App/data/02-preprocessed"
os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(out_dir, "ee_data_clean.csv")
df_clean.to_csv(out_path, index=False)
print(f"Saved cleaned dataset → {out_path}")

Saved cleaned dataset → /Users/jamesantoarnoldj/Desktop/Projects/EE App/data/02-preprocessed/ee_data_clean.csv
